# ClimateVision — Flood Model Training (Sen1Floods11)

Run top to bottom on a **GPU runtime** (Runtime → Change runtime type → GPU).
This trains a real flood-detection U-Net and exports it for deployment.

**Prerequisite:** the latest scripts must be on `main` (push from your laptop first):
`download_datasets.py`, `prepare_sen1floods11.py`, `train_real.py`, and the `dataset.py` change.


## 1. Setup — clone repo, install, check GPU


In [ ]:
!git clone https://github.com/Climate-Vision/ClimateVision.git
%cd ClimateVision
!git pull origin main          # make sure newest scripts are present
!pip install -q -r requirements.txt
!pip install -q -e .
import torch; print('CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')


In [ ]:
# Sanity check the scripts exist (fail early if not pushed yet)
import os
need = ['scripts/prepare_sen1floods11.py','scripts/train_real.py','scripts/export_model.py']
missing = [p for p in need if not os.path.exists(p)]
assert not missing, f'Missing (push these first): {missing}'
print('All scripts present.')


## 2. Persist to Google Drive (so checkpoints survive a runtime reset)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/climatevision/models


## 3. Authenticate to Google Cloud (for the Sen1Floods11 bucket)


In [ ]:
from google.colab import auth
auth.authenticate_user()
PROJECT = 'kinos-473422'   # your GCP/GEE project
!gcloud config set project {PROJECT}


## 4. Download Sen1Floods11 hand-labeled data (~hundreds of chips)


In [ ]:
!mkdir -p data/sen1floods11
!gcloud storage cp --recursive gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand    data/sen1floods11/
!gcloud storage cp --recursive gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand  data/sen1floods11/
!gcloud storage cp --recursive gs://sen1floods11/v1.1/splits                                    data/sen1floods11/
print('S2:', len(os.listdir('data/sen1floods11/S2Hand')), '| Labels:', len(os.listdir('data/sen1floods11/LabelHand')))


## 5. Convert to the ClimateVision training layout
Extracts S2 bands B03,B08,B11 and pairs masks. Add `--jrc-dir` only for the 3-class variant.


In [ ]:
!python scripts/prepare_sen1floods11.py \
  --s2-dir data/sen1floods11/S2Hand \
  --label-dir data/sen1floods11/LabelHand \
  --splits-dir data/sen1floods11/splits \
  --out-dir data/datasets/flooding


## 6. Train
Watch `val_iou` climb. Early stopping is automatic. ~1–3h depending on GPU.


In [ ]:
!python scripts/train_real.py --analysis-type flooding \
  --data-dir data/datasets/flooding \
  --epochs 50 --batch-size 8 --image-size 256 \
  --out /content/drive/MyDrive/climatevision/models


## 7. Locate the best checkpoint


In [ ]:
import glob
runs = sorted(glob.glob('/content/drive/MyDrive/climatevision/models/flooding_*/best_model.pth'))
assert runs, 'No checkpoint found — did training finish?'
CKPT = runs[-1]; print('Best checkpoint:', CKPT)


## 8. Evaluate + governance gate (promote only if it passes)


In [ ]:
!python scripts/evaluate.py --checkpoint "$CKPT" --data-dir data/datasets/flooding || true
!python scripts/governance_ci_gate.py || true


## 9. Export to ONNX (what the API serves)


In [ ]:
!python scripts/export_model.py --checkpoint "$CKPT"
import os; d=os.path.dirname(CKPT); print('Artifacts in', d, '->', os.listdir(d))


## 10. Download the model, then deploy

Download `best_model.pth` and `model.onnx` from the run folder above, then on your laptop:
```bash
cp <downloaded>/* models/flooding_<date>/
git add models/flooding_<date>/
git commit -m 'feat(models): trained Sen1Floods11 flood model'
git push origin main          # triggers Render rebuild
```
Verify it's live, then drop the preview label:
```bash
curl -s https://climatevision.green/api/health/models | jq
```


In [ ]:
from google.colab import files
import os
d = os.path.dirname(CKPT)
for f in ['best_model.pth','model.onnx']:
    p = os.path.join(d,f)
    if os.path.exists(p): files.download(p)
